# QueryMind - Olist Data Exploration

Exploring the Olist Brazilian E-Commerce dataset to understand schema structure, data quality and join relationships.
Findings here directly inform the schema metadata YAML and business glossary for the RAG layer.

In [1]:
import sys
from pathlib import Path

# Add project root to Python's module search path - so that we can import src.*
# Standard practice for notebooks living inside a subdirectory
project_root = str(Path.cwd().parent) # notebooks/ -> querymind/
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")

Project root: c:\Users\User\Desktop\QueryMind\querymind


In [2]:
# Imports
import pandas as pd
from sqlalchemy import create_engine, text, inspect

from src.database.connection import get_engine

engine = get_engine(readonly=True)
print("Connected Successfully!")

Connected Successfully!


In [3]:
inspector = inspect(engine)
tables = inspector.get_table_names()

print(f"Tables in database: {len(tables)}\n")
for t in sorted(tables):
    print(f"  • {t}")

Tables in database: 9

  • olist_customers
  • olist_geolocation
  • olist_order_items
  • olist_order_payments
  • olist_order_reviews
  • olist_orders
  • olist_products
  • olist_sellers
  • product_category_name_translation


In [4]:
with engine.connect() as conn:
    row_counts = {}
    for table in sorted(tables):
        count = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        row_counts[table] = count

row_counts_df = pd.DataFrame(
    row_counts.items(), columns=["table_name", "row_count"]
).sort_values("row_count", ascending=False)

row_counts_df["row_count"] = row_counts_df["row_count"].apply(lambda x: f"{x:,}")
row_counts_df

,table_name,row_count
1,olist_geolocation,"1,000,163"
2,olist_order_items,"112,650"
3,olist_order_payments,"103,886"
0,olist_customers,"99,441"
5,olist_orders,"99,441"
4,olist_order_reviews,"99,224"
6,olist_products,"32,951"
7,olist_sellers,"3,095"
8,product_category_name_translation,71


* There are 99,441 orders and 112,650 order items.
    * Each order can contain multiple items.
    * Therefore, average order has 1.13 items.

* 71 entries of Category name translations.
    * Brazilian marketplace - Portuguese category names.
    * This table maps each Portuguese category name to its English equivalent.


In [5]:
for table in sorted(tables):
    columns = inspector.get_columns(table)
    print(f"\n{'=' * 60}")
    print(f"  {table} ({row_counts[table]:,} rows)")
    print(f"{'=' * 60}")
    for col in columns:
        nullable = "NULL ok" if col["nullable"] else "NOT NULL"
        print(f"  {col['name']:<45} {str(col['type']):<15} {nullable}")


  olist_customers (99,441 rows)
  customer_id                                   TEXT            NULL ok
  customer_unique_id                            TEXT            NULL ok
  customer_zip_code_prefix                      BIGINT          NULL ok
  customer_city                                 TEXT            NULL ok
  customer_state                                TEXT            NULL ok

  olist_geolocation (1,000,163 rows)
  geolocation_zip_code_prefix                   BIGINT          NULL ok
  geolocation_lat                               FLOAT           NULL ok
  geolocation_lng                               FLOAT           NULL ok
  geolocation_city                              TEXT            NULL ok
  geolocation_state                             TEXT            NULL ok

  olist_order_items (112,650 rows)
  order_id                                      TEXT            NULL ok
  order_item_id                                 BIGINT          NULL ok
  product_id                 

* Notice SQLite does not have DATETIME type - dates are stored as TEXT.
    * Should still be in ISO timestamp format - verify.

* All columns are nullable.

In [6]:
# Display 'head' of each table - for better understanding of structure.
with engine.connect() as conn:
    for table in sorted(tables):
        print(f"\n{'=' * 80}")
        print(f"  {table}")
        print(f"\n{'=' * 80}")
        df = pd.read_sql(f"SELECT * FROM {table} LIMIT 5", conn)
        display(df)


  olist_customers



,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



  olist_geolocation



,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
3,1041,-23.544392,-46.639499,sao paulo,SP
4,1035,-23.541578,-46.641607,sao paulo,SP



  olist_order_items



...

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0



  olist_sellers



,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP



  product_category_name_translation



,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


## NULL Analysis
Identify which columns have missing data, and how much?
Should directly inform schema metadata - LLM needs to know about NULLs to generate appropriate queries
(e.g. use COALESCE or explicitly filtering NULLs)

In [7]:
# NULL counts across all tables
with engine.connect() as conn:
    for table in sorted(tables):
        df = pd.read_sql(f"SELECT * FROM {table}", conn)
        null_counts = df.isnull().sum()
        null_counts = null_counts[null_counts > 0]

        print(f"\n{'=' * 60}")
        print(f"  {table} ({len(df):,} rows)")
        print(f"{'=' * 60}")

        if null_counts.empty:
            print("  No NULLs found!")
        else:
            for col, count in null_counts.items():
                pct = count / len(df) * 100
                print(f"  {col:<45} {count:>7,} ({pct:>5.1f}%)")


  olist_customers (99,441 rows)
  No NULLs found!

  olist_geolocation (1,000,163 rows)
  No NULLs found!

  olist_order_items (112,650 rows)
  No NULLs found!

  olist_order_payments (103,886 rows)
  No NULLs found!

  olist_order_reviews (99,224 rows)
  review_comment_title                           87,656 ( 88.3%)
  review_comment_message                         58,247 ( 58.7%)

  olist_orders (99,441 rows)
  order_approved_at                                 160 (  0.2%)
  order_delivered_carrier_date                    1,783 (  1.8%)
  order_delivered_customer_date                   2,965 (  3.0%)

  olist_products (32,951 rows)
  product_category_name                             610 (  1.9%)
  product_name_lenght                               610 (  1.9%)
  product_description_lenght                        610 (  1.9%)
  product_photos_qty                                610 (  1.9%)
  product_weight_g                                    2 (  0.0%)
  product_length_cm              

Findings:
* Reviews are mostly text-free. ~88% have no title & ~59% have no comment message.
* When filtering on reviews, stick to numeric review_score.
___
* Orders have low NULL numbers - perhaps indicating of orders which were canceled, still in shipping and or somehow lost.
* Therefore, delivery time calculations, should only filter where order_status = 'delivered' - in order to avoid NULLs.

___
* Products, once again, have low NULL numbers - perhaps indicating of placeholder and or incomplete product listings.

## Cardinality Analysis
How many unique values per column? 

High cardinality = IDs and keys

Low cardinality = categories and statuses (good for GROUP BY / filtering)

In [8]:
with engine.connect() as conn:
    for table in sorted(tables):
        df = pd.read_sql(f"SELECT * FROM {table}", conn)

        print(f"\n{'=' * 60}")
        print(f"  {table} ({len(df):,} rows)")
        print(f"{'=' * 60}")
        
        for col in df.columns:
            nunique = df[col].nunique()
            ratio = nunique / len(df) * 100
            # Flag columns that look like categorical (low unique count)
            marker = "  ◄ categorical" if nunique <= 30 else ""
            print(f"  {col:<45} {nunique:>8,} unique ({ratio:>5.1f}%){marker}")


  olist_customers (99,441 rows)
  customer_id                                     99,441 unique (100.0%)
  customer_unique_id                              96,096 unique ( 96.6%)
  customer_zip_code_prefix                        14,994 unique ( 15.1%)
  customer_city                                    4,119 unique (  4.1%)
  customer_state                                      27 unique (  0.0%)  ◄ categorical

  olist_geolocation (1,000,163 rows)
  geolocation_zip_code_prefix                     19,015 unique (  1.9%)
  geolocation_lat                                717,360 unique ( 71.7%)
  geolocation_lng                                717,613 unique ( 71.7%)
  geolocation_city                                 8,011 unique (  0.8%)
  geolocation_state                                   27 unique (  0.0%)  ◄ categorical

  olist_order_items (112,650 rows)
  order_id                                        98,666 unique ( 87.6%)
  order_item_id                                       21 uni

## Categorical Value Distribution
Actual values and frequencies for low-cardinality columns.

These will be documented in schema_metadata.yaml, so that LLM knows valid filter values.

In [9]:
# Columns we want to inspect - picked from cardinality analysis
categorical_columns = {
    "olist_customers": ["customer_state"],
    "olist_geolocation": ["geolocation_state"],
    "olist_order_items": ["order_item_id"],
    "olist_order_payments": ["payment_sequential", "payment_type", "payment_installments"],
    "olist_order_reviews": ["review_score"],
    "olist_orders": ["order_status"],
    "olist_products": ["product_photos_qty"],
    "olist_sellers": ["seller_state"]
}

with engine.connect() as conn:
    for table, columns in categorical_columns.items():
        for col in columns:
            df = pd.read_sql(
                f"SELECT {col}, COUNT(*) as count FROM {table} GROUP BY {col} ORDER BY count DESC",
                conn,
            )
            total = df["count"].sum()
            df["pct"] = (df["count"] / total * 100).round(1)

            print(f"\n{'=' * 60}")
            print(f"  {table}.{col}")
            print(f"{'=' * 60}")
            for _, row in df.iterrows():
                bar = "█" * int(row["pct"] / 2)  # Simple visual bar
                print(f"  {str(row[col]):<25} {row['count']:>8,} ({row['pct']:>5.1f}%) {bar}")



  olist_customers.customer_state
  SP                          41,746 ( 42.0%) █████████████████████
  RJ                          12,852 ( 12.9%) ██████
  MG                          11,635 ( 11.7%) █████
  RS                           5,466 (  5.5%) ██
  PR                           5,045 (  5.1%) ██
  SC                           3,637 (  3.7%) █
  BA                           3,380 (  3.4%) █
  DF                           2,140 (  2.2%) █
  ES                           2,033 (  2.0%) █
  GO                           2,020 (  2.0%) █
  PE                           1,652 (  1.7%) 
  CE                           1,336 (  1.3%) 
  PA                             975 (  1.0%) 
  MT                             907 (  0.9%) 
  MA                             747 (  0.8%) 
  MS                             715 (  0.7%) 
  PB                             536 (  0.5%) 
  PI                             495 (  0.5%) 
  RN                             485 (  0.5%) 
  AL                            

Findings (for Cardinality & Categorical distributions):
* customer_id (99,441) vs. customer_unique_id (96,096) indicates there are 3,345 customers who placed more than one orderm but got a different ID each time.
* Therefore, should use customer_unique_id, in order to track returning-customers and repeat-purchases.
___
* order_item_id seems to be a sequence number, NOT an ID.
* 87.6% of values being 1 confirms this. This column simply lists the item number, within each order - i.e. 1st item, 2nd item, 3rd item, etc.
* Maximum value being 21 means one order had 21 items.
* Similar takeaway for payment_sequential - 95.6% of values are 1, indicating most orders had a single method of payment.
    * Some orders split payments across multiple methods. Amazingly, this goes up to 29 payments.

___
* Geolocation has 19,015 unique zip code prefixes, across 1 million rows.
    * Translates to ~53 coordinate points per zip code on average. 1:Many relationship.

##### Distribution breakdown:
___
* 97% of orders have status 'delivered'.
    * Queries regarding revenue and or delivery times should still filter on WHERE order_status = 'delivered'.

___
* Credit card makes up for ~74% of purchases. 
* Boleto (a Brazilian payment method - basically a bank slip) at ~19%, will need to be clearly outlined for the LLM.
* 3 payments classified as 'not_defined' can simply be considered noise.

___
* Review scores are heavily skewed toward 5 stars (~58%)

___
* Sao Paolo (SP) makes up for the majority of customers (~42%) and sellers (~60%). After all, it is Brazil's most populous state.
* The LLM should know, the 27 states listed are Brazil's 26 states plus the Federal District (DF).

## Date Range Analysis
What time period does the data cover?

Should inform the business glossary (e.g., "Data covers Sept 2016 - Oct 2018").

Helps LLM avoid generating queries for time periods with no data.

In [10]:
date_columns = {
    "olist_orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
    "olist_order_reviews": [
        "review_creation_date",
        "review_answer_timestamp",
    ],
}

with engine.connect() as conn:
    for table, columns in date_columns.items():
        print(f"\n{'=' * 60}")
        print(f"  {table}")
        print(f"{'=' * 60}")
        for col in columns:
            result = conn.execute(text(
                f"SELECT MIN({col}), MAX({col}) FROM {table} WHERE {col} IS NOT NULL"
            ))
            min_val, max_val = result.fetchone()
            print(f"  {col:<40} {min_val}  →  {max_val}")


  olist_orders
  order_purchase_timestamp                 2016-09-04 21:15:19  →  2018-10-17 17:30:18
  order_approved_at                        2016-09-15 12:16:38  →  2018-09-03 17:40:06
  order_delivered_carrier_date             2016-10-08 10:34:01  →  2018-09-11 19:48:28
  order_delivered_customer_date            2016-10-11 13:46:32  →  2018-10-17 13:22:46
  order_estimated_delivery_date            2016-09-30 00:00:00  →  2018-11-12 00:00:00

  olist_order_reviews
  review_creation_date                     2016-10-02 00:00:00  →  2018-08-31 00:00:00
  review_answer_timestamp                  2016-10-07 18:32:28  →  2018-10-29 12:27:35


In [11]:
# Monthly order volume --> identifies where data is dense vs. sparse
query = """
    SELECT 
        SUBSTR(order_purchase_timestamp, 1, 7) AS month,
        COUNT(*) AS order_count
    FROM olist_orders
    GROUP BY SUBSTR(order_purchase_timestamp, 1, 7)
    ORDER BY month
"""

with engine.connect() as conn:
    monthly = pd.read_sql(query, conn)

print("Monthly order volume:\n")
for _, row in monthly.iterrows():
    bar = "█" * (row["order_count"] // 200)
    print(f"  {row['month']}  {row['order_count']:>6,}  {bar}")

Monthly order volume:

  2016-09       4  
  2016-10     324  █
  2016-12       1  
  2017-01     800  ████
  2017-02   1,780  ████████
  2017-03   2,682  █████████████
  2017-04   2,404  ████████████
  2017-05   3,700  ██████████████████
  2017-06   3,245  ████████████████
  2017-07   4,026  ████████████████████
  2017-08   4,331  █████████████████████
  2017-09   4,285  █████████████████████
  2017-10   4,631  ███████████████████████
  2017-11   7,544  █████████████████████████████████████
  2017-12   5,673  ████████████████████████████
  2018-01   7,269  ████████████████████████████████████
  2018-02   6,728  █████████████████████████████████
  2018-03   7,211  ████████████████████████████████████
  2018-04   6,939  ██████████████████████████████████
  2018-05   6,873  ██████████████████████████████████
  2018-06   6,167  ██████████████████████████████
  2018-07   6,292  ███████████████████████████████
  2018-08   6,512  ████████████████████████████████
  2018-09      16  
  2018-10

Findings:
* Data covers September 2016 to October 2018.
* 'Usable' data is essentially from January 2017 to August 2018
* Once again, valuable information for our business glossary.
    * If the end user asks about 2016 revenue (for instance), the LLM should explain that only partial, likely meaningless data exists.

## Join Validation
Verify every foreign key relationship in the schema.

Orphan records (child rows with no matching parent) would cause JOINs to silently lose data.

Highly critical to document, for the LLM's schema context.

In [12]:
# Each tuple: (child_table, child_column, parent_table, parent_column)
relationships = [
    ("olist_orders",         "customer_id",    "olist_customers",    "customer_id"),
    ("olist_order_items",    "order_id",       "olist_orders",       "order_id"),
    ("olist_order_items",    "product_id",     "olist_products",     "product_id"),
    ("olist_order_items",    "seller_id",      "olist_sellers",      "seller_id"),
    ("olist_order_payments", "order_id",       "olist_orders",       "order_id"),
    ("olist_order_reviews",  "order_id",       "olist_orders",       "order_id"),
    ("olist_products",       "product_category_name",
     "product_category_name_translation", "product_category_name"),
]

with engine.connect() as conn:
    print(f"{'Child':<30} {'Column':<30} {'Parent':<35} {'Orphans':>10}")
    print("─" * 110)

    for child_tbl, child_col, parent_tbl, parent_col in relationships:
        result = conn.execute(text(f"""
            SELECT COUNT(*) FROM {child_tbl} c
            WHERE c.{child_col} IS NOT NULL
              AND c.{child_col} NOT IN (SELECT {parent_col} FROM {parent_tbl})
        """))
        orphan_count = result.scalar()
        status = "✓ clean" if orphan_count == 0 else f"⚠ {orphan_count:,} orphans"
        print(f"  {child_tbl:<28} {child_col:<30} {parent_tbl:<33} {status:>10}")

Child                          Column                         Parent                                 Orphans
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
  olist_orders                 customer_id                    olist_customers                      ✓ clean
  olist_order_items            order_id                       olist_orders                         ✓ clean
  olist_order_items            product_id                     olist_products                       ✓ clean
  olist_order_items            seller_id                      olist_sellers                        ✓ clean
  olist_order_payments         order_id                       olist_orders                         ✓ clean
  olist_order_reviews          order_id                       olist_orders                         ✓ clean
  olist_products               product_category_name          product_category_name_translation ⚠ 13 orphans


In [13]:
# Verify critical multi-hop join path (orders --> items --> products --> categories)
query = """
    SELECT 
        COUNT(DISTINCT o.order_id)    AS orders,
        COUNT(oi.order_item_id)       AS items,
        COUNT(DISTINCT p.product_id)  AS products,
        COUNT(DISTINCT t.product_category_name_english) AS categories
    FROM olist_orders o
    JOIN olist_order_items oi ON o.order_id = oi.order_id
    JOIN olist_products p ON oi.product_id = p.product_id
    LEFT JOIN product_category_name_translation t 
        ON p.product_category_name = t.product_category_name
"""

with engine.connect() as conn:
    result = pd.read_sql(query, conn)
    display(result)
    print("\nThis is the core join path for most business queries.")
    print("LEFT JOIN on translation because some products may lack a category.")

,orders,items,products,categories
0,98666,112650,32951,71



This is the core join path for most business queries.
LEFT JOIN on translation because some products may lack a category.


Findings:
* Joins are clean EXCEPT products --> category - 13 orphans.
    * 13 products have a category name, which does not exist in the translation table.
    * Indicates that the LEFT JOIN in multi-hop is the appropriate choice (rather than INNER JOIN). Avoids silently dropping records.
    * This information should be fed directly into the schema metadata documents.

In [14]:
# Identify Orphan Categories - Join Validation
query = """
    SELECT DISTINCT p.product_category_name
    FROM olist_products p
    WHERE p.product_category_name IS NOT NULL
      AND p.product_category_name NOT IN (
          SELECT product_category_name 
          FROM product_category_name_translation
      )
"""

with engine.connect() as conn:
    orphans = pd.read_sql(query, conn)
    print(f"Product categories without English translation ({len(orphans)}):\n")
    for _, row in orphans.iterrows():
        print(f"  • {row['product_category_name']}")

Product categories without English translation (2):

  • pc_gamer
  • portateis_cozinha_e_preparadores_de_alimentos


Only 2 untranslated categories.
* pc_gamer --> self explanatory
* portateis_cozinha_e_preparadores_de_alimentos --> loosely translated to "Portable kitchen and food preparers".

Both are quite minor and can be clearly outlined in the schema metadata.

In [15]:
# customer_id vs customer_unique_id relationship - Cardinality
query = """
    SELECT customer_unique_id, COUNT(*) AS order_count
    FROM olist_customers
    GROUP BY customer_unique_id
    HAVING COUNT(*) > 1
    ORDER BY order_count DESC
    LIMIT 10
"""

with engine.connect() as conn:
    repeat_customers = pd.read_sql(query, conn)
    print("Top 10 repeat customers (by number of orders):\n")
    display(repeat_customers)
    
    total_unique = conn.execute(text(
        "SELECT COUNT(DISTINCT customer_unique_id) FROM olist_customers"
    )).scalar()
    total_rows = conn.execute(text(
        "SELECT COUNT(*) FROM olist_customers"
    )).scalar()
    repeat = conn.execute(text("""
        SELECT COUNT(*) FROM (
            SELECT customer_unique_id FROM olist_customers 
            GROUP BY customer_unique_id HAVING COUNT(*) > 1
        )
    """)).scalar()
    
    print(f"\n{total_rows:,} customer rows → {total_unique:,} unique customers")
    print(f"{repeat:,} customers placed more than one order ({repeat/total_unique*100:.1f}%)")

Top 10 repeat customers (by number of orders):



,customer_unique_id,order_count
0,8d50f5eadf50201ccdcedfb9e2ac8455,17
1,3e43e6105506432c953e165fb2acf44c,9
2,ca77025e7201e3b30c44b472ff346268,7
3,6469f99c1f9dfae7733b25662e7f1782,7
4,1b6c7548a2a1f9037c1fd3ddfed95f33,7
5,f0e310a6839dce9de1638e0fe5ab282a,6
6,de34b16117594161a6a89c50b289d35a,6
7,dc813062e0fc23409cd255f7f53c7074,6
8,63cfc61cee11cbe306bff5857d00bfe4,6
9,47c1a3033b8b77b3ab6e109eb4d5fdf3,6



99,441 customer rows → 96,096 unique customers
2,997 customers placed more than one order (3.1%)


Here, we understand that the Olist dataset is predominantly made up of single-purchase customers. Repeat customers are rare (~3.1%), where the most active user placed 17 orders.

Whereas concepts like 'repeat purchase rates' or 'customer lifetime value' are vital for some businesses, however here, the LLM must understand that the vast majority of customers are one-time buyers.

Also means that customer_unique_id and customer_id will largely return identical counts, but the distinction still matters for absolute correctness.

___

___

## Summary of Key Findings

**Scope of Data**
* 9 tables
* ~1.5M total rows
* Reliable data range: **January 2017 - August 2018**
    * Data before Jan '17 and after Aug '18 is sparse/incomplete

**Core Business Data**
* ~99k orders
* ~113k singular items
* ~33k products
* ~3k sellers

**Schema & Relationships**
* All foreign keys joins are clean
    * Except `products --> category translation`; 2 untranslated categories
* Core join path: `orders --> order_items --> products --> category_translation`
    * Use LEFT JOIN on translation, to ensure rows without a translation do not get dropped
* `customer_id` is per-order
* `customer_unique_id` - true customer identifier - 96,096 unique customers
* `order_item_id` - sequence number within each order; NOT unique row ID
* `payment_sequential` - sequence number within each order's methods of payment

**Data Types & Quality**
* Date columns stored as TEXT (SQLite has no native datetime alternative)
    * Still maintaining ISO format
* Reviews: 88% have no title, 59% have no comment
    * Review analysis must focus on `review_score`
* Products: 610 (1.9%) of all products have no category and or descriptive metadata
* Orders: 3% have no delivery dates - non-delivered, likely canceled/shipping/processing

**Noteworthy Distributions**
* Order status: 97% delivered. Queries relating to revenue/delivery should filter `WHERE order_status = 'delivered`
* Payment: 74% credit card, 19% Boleto (Brazilian bank slip), 6% voucher, 1.5% debit
* Reviews: Skewed positive - 58% five-star, 11.5% one-star
* Geography: Sao Paulo (SP) dominates - 42% of customers, 60% of sellers

**Implications for RAG Layer**
* Schema metadata must address TEXT-typed date columns - including their ISO format
* Business glossary checklist:
    * Revenue definition (price + freight)
    * Filtered to delivered orders only
    * Relevant/reliable date range
    * customer_unique_id vs customer_id
    * Boleto explanation
    * Context for certain peaks in the year - e.g. Black Friday in November, Christmas/New Years in December-January
* Example queries should cover full join path, alongside data & date filtering patterns